# Colab writing-model experiment: OpenStax + Amplify

Select **Runtime → Change runtime type → T4 GPU**, then **Run all**. The upload cell will request `Student_Work_Collection__Essay_Scoring.pdf`. Training starts after that upload; no Claude API key, paid GPU purchase, Drive mount, or model publication is used.

The training code is embedded and checksum-verified. The Amplify PDF is **not published or embedded**; upload the same PDF in your own Colab session. The user authorized its use and confirmed authority in chat on October 17, 2026; this is an experiment, not a commercial deployment.

Training combines historical OpenStax commentary with all 41 distinct Amplify writing samples and two user-provided preference pairs using their original rubrics. Original pages resolve the repeated-page scoring conflicts. The nine mastery essays train native mastery commentary without inventing assignment prompts; duplicate versions are not extra examples. Your six-category rubric is supplied as context, with missing ratings left missing. **This is not your six-category grader**: native publisher labels and qualitative commentary are distinct tasks. Eight narrative examples are included, all in training, so this run cannot establish narrative holdout accuracy.

The trained adapter downloads before held-out comparison; a final ZIP includes base-versus-adapter outputs and native-score metrics. No quality claim is made until those results are reviewed.


In [ ]:
import subprocess, pathlib, sys, os, io, zipfile, base64, hashlib, json
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime → Change runtime type → T4 GPU. No model download or training started.")
print("GPU:", torch.cuda.get_device_name(0))
repo = pathlib.Path("/content/essaygrader-colab-run") if pathlib.Path("/content").exists() else pathlib.Path.cwd() / "essaygrader-colab-run"
if repo.exists():
    raise RuntimeError("Run folder already exists. Download prior results, then use a fresh runtime.")
source_zip = base64.b64decode('...')
expected_source_sha256 = '08e3a01c8195b7db5194dd91be07c02e142cb5ae1749d8a0d7742601e2b0fd20'
assert hashlib.sha256(source_zip).hexdigest() == expected_source_sha256, "Embedded training code checksum mismatch"
repo.mkdir()
with zipfile.ZipFile(io.BytesIO(source_zip)) as archive:
    for member in archive.infolist():
        target = (repo / member.filename).resolve()
        if not target.is_relative_to(repo.resolve()):
            raise RuntimeError("Invalid source archive path")
    archive.extractall(repo)
os.chdir(repo)
pathlib.Path("code-snapshot.json").write_text(json.dumps({"sha256": expected_source_sha256, "files": sorted(str(p.relative_to(repo)) for p in repo.rglob("*") if p.is_file())}, indent=2))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-training.txt"], check=True)
print("Training dependencies installed. Source snapshot:", expected_source_sha256)


In [ ]:
import shutil
if shutil.which("pdftotext") is None:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "poppler-utils"], check=True)
from google.colab import files
print("Upload Student_Work_Collection__Essay_Scoring.pdf (the same 120-page document).")
uploaded = files.upload()
from model_work.amplify_native import PDF_SHA256
candidates=[pathlib.Path(name) for name, content in uploaded.items() if hashlib.sha256(content).hexdigest()==PDF_SHA256]
if len(candidates)!=1:
    raise RuntimeError("Upload the exact reviewed PDF; no training started. Other revisions require extraction review.")
amplify_pdf=candidates[0].resolve()
print("Verified uploaded PDF checksum.")

from urllib.request import urlopen
preference_url="https://raw.githubusercontent.com/sfstast/essaygrader/8d05ea2be59aa920c1e04216566109835eab83ad/model_work/examples/user_style_preferences.json"
with urlopen(preference_url, timeout=30) as response:
    preference_bytes=response.read()
if hashlib.sha256(preference_bytes).hexdigest()!="f69be2d4e472a11ebf6f20cbd88d0f65f5adab380f2dcac043c2c13ed177a271":
    raise RuntimeError("Published preference-file checksum mismatch.")
preference_file=repo/"model_work/examples/user_style_preferences.json"
preference_file.parent.mkdir(parents=True,exist_ok=True)
preference_file.write_bytes(preference_bytes)
print("Loaded the two unchanged preference pairs already present in the repository.")


## Collect the licensed teaching examples

Upload the Amplify PDF in the preceding cell. The OpenStax collector pins the historical edition, retains license/attribution and excludes quote-containing pairs conservatively. All excerpts from an author/essay remain in one split. The test split stays outside training. Do not replace the source with the current noncommercial edition.


In [ ]:
subprocess.run([sys.executable,"-m","model_work.openstax_annotations","download","--out","model_work/data/public/free-gpu-source"],check=True)
subprocess.run([sys.executable,"-m","model_work.openstax_annotations","prepare","--source","model_work/data/public/free-gpu-source","--out","model_work/prepared/free-gpu-annotations"],check=True)
subprocess.run([sys.executable,"-m","model_work.mixed_writing","--annotations","model_work/prepared/free-gpu-annotations","--pdf",str(amplify_pdf),"--out","model_work/prepared/free-gpu-writing"],check=True)
subprocess.run([sys.executable,"-m","model_work.train_lora","--mixed-writing","--prepared","model_work/prepared/free-gpu-writing"],check=True)


## Resolve a model revision and train

The model identifier alone is not license proof. Resolve its exact revision, then the trainer verifies the pinned model card and Apache-2.0 LICENSE before loading weights. A missing or different license stops the run. One epoch and 4-bit QLoRA keep this an inexpensive experiment; free runtime timeouts may interrupt it.


In [ ]:
import json
from huggingface_hub import HfApi
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
revision = HfApi().model_info(model_id).sha
pathlib.Path("free-run-model.json").write_text(json.dumps({"model":model_id,"revision":revision},indent=2))
subprocess.run([sys.executable,"-m","model_work.train_lora","--mixed-writing","--prepared","model_work/prepared/free-gpu-writing","--train","--model",model_id,"--revision",revision,"--out","model_work/adapters/free-commentary-v1","--qlora","--epochs","1","--max-length","4096"],check=True)


In [ ]:
import shutil
checkpoint_dir=pathlib.Path("free-commentary-checkpoint")
checkpoint_dir.mkdir(exist_ok=False)
shutil.copytree("model_work/adapters/free-commentary-v1",checkpoint_dir/"training")
for filename in ("ATTRIBUTION.txt", "LICENSE", "manifest.json", "AMPLIFY_PERMISSION.txt"):
    shutil.copy2(pathlib.Path("model_work/prepared/free-gpu-writing")/filename,checkpoint_dir/filename)
for filename in ("free-run-model.json", "code-snapshot.json"):
    shutil.copy2(filename,checkpoint_dir/filename)
checkpoint_archive=shutil.make_archive("free-commentary-checkpoint","zip",checkpoint_dir)
print("Training finished; adapter saved. Quality review remains pending.")
try:
    from google.colab import files
    files.download(checkpoint_archive)
except ImportError:
    print("Checkpoint archive:",checkpoint_archive)


## Compare base and tuned commentary on unseen excerpts

This comparison is a review packet, not an automatic quality or grading pass. Read each answer against the original editorial comment. Check accuracy, useful explanation, unsupported claims and unnecessary rewriting. Both models receive the same input. Do not tune on these test examples and then call them an independent test.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from model_work.mixed_writing import preflight, messages
_, splits = preflight("model_work/prepared/free-gpu-writing")
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, trust_remote_code=False)
base = AutoModelForCausalLM.from_pretrained(model_id, revision=revision, trust_remote_code=False,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.float16),device_map="auto")
model = PeftModel.from_pretrained(base,"model_work/adapters/free-commentary-v1/adapter")
model.eval()
def response(row, tuned):
    inputs=tokenizer.apply_chat_template(messages(row)[:-1],add_generation_prompt=True,return_tensors="pt").to(model.device)
    with torch.inference_mode():
        if tuned:
            result=model.generate(inputs,attention_mask=torch.ones_like(inputs),max_new_tokens=384,do_sample=False,pad_token_id=tokenizer.eos_token_id)
        else:
            with model.disable_adapter():
                result=model.generate(inputs,attention_mask=torch.ones_like(inputs),max_new_tokens=384,do_sample=False,pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(result[0,inputs.shape[1]:],skip_special_tokens=True)
reviews=[]
for row in splits["test"]:
    prompt=messages(row)
    reviews.append({"id":row["id"],"task_family":row["task_family"],"input_messages":prompt[:-1],
        "published_target":prompt[-1]["content"],"base_response":response(row,False),
        "tuned_response":response(row,True),"source_url":row["source_url"],"human_review":None,
        "native_score":row.get("native_score"),"native_criterion":row.get("native_criterion")})
pathlib.Path("free-commentary-review.json").write_text(json.dumps(reviews,indent=2,ensure_ascii=False))
def score_from_text(text):
    text=text.strip()
    if text.startswith("```"):
        text=text.split("\n",1)[-1].rsplit("```",1)[0].strip()
    try:
        value=json.loads(text).get("score")
        return value if type(value) is int and 1<=value<=4 else None
    except (ValueError,AttributeError):return None
native=[r for r in reviews if r["task_family"]=="amplify_native"]
metrics={"teacher_calibrated":False,"narrative_holdout_evaluated":False,"interpretation":"Native-score agreement on one held-out assignment only; not six-category rubric accuracy."}
for label in ("base","tuned"):
    pairs=[(r["native_score"],score_from_text(r[label+"_response"])) for r in native]
    valid=[(a,b) for a,b in pairs if b is not None]
    metrics[label]={"expected":len(pairs),"valid_outputs":len(valid),"invalid_outputs":len(pairs)-len(valid),
        "mae_valid_only":sum(abs(a-b) for a,b in valid)/len(valid) if valid else None,
        "exact_agreement_valid_only":sum(a==b for a,b in valid)/len(valid) if valid else None}
pathlib.Path("native-score-metrics.json").write_text(json.dumps(metrics,indent=2))
print(json.dumps(metrics,indent=2))
print("Prepared",len(reviews),"held-out comparisons. Human quality review is pending.")


## Download your results

The adapter requires the same base model/revision to run. Keep attribution with the adapter. The download includes an unfilled human-review packet, license evidence and code/package versions. Nothing is installed into the live website automatically.


In [ ]:
import shutil
bundle=pathlib.Path("free-commentary-download");bundle.mkdir(exist_ok=False)
shutil.copytree("model_work/adapters/free-commentary-v1",bundle/"training")
for file in ("ATTRIBUTION.txt","LICENSE","manifest.json","AMPLIFY_PERMISSION.txt"):
    shutil.copy2(pathlib.Path("model_work/prepared/free-gpu-writing")/file,bundle/file)
for file in ("free-run-model.json","free-commentary-review.json","code-snapshot.json","native-score-metrics.json"):
    shutil.copy2(file,bundle/file)
(bundle/"packages.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True))
(bundle/"README.txt").write_text("Experimental mixed writing adapter. OpenStax commentary plus Amplify native rubric scores. Requires the pinned base model. No validated numeric grades or site deployment. Retain attribution and review held-out outputs before use.\n")
archive=shutil.make_archive("free-commentary-results","zip",bundle)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Download from notebook files:",archive)
